# AGENT WEB · Colab Blender 4.x
Google ColabのGPUランタイム（L4またはT4）でBlenderレンダリングを実行するための接続用ノートブックです。
AGENT WEBからの要求を受け取り、Cycles GPUまたはEeveeによる静止画・アニメーションレンダリングを行います。

In [ ]:
# 1. Blender Linux 4.2 LTS のセットアップ
!apt-get -qq update && apt-get -qq install -y libgl1 libegl1 libxi6 libxkbcommon0
!if [ ! -d "/content/blender-4.2.3-linux-x64" ]; then \
    wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz && \
    tar -xf blender-4.2.3-linux-x64.tar.xz -C /content/; \
fi
!ln -sf /content/blender-4.2.3-linux-x64/blender /usr/local/bin/blender
!blender --version

In [ ]:
# 2. GPU (CUDA/OptiX) 有効化の検証
import subprocess
enable_gpu_script = """
import bpy
prefs = bpy.context.preferences.addons['cycles'].preferences
prefs.get_devices()
for dev_type in ('OPTIX', 'CUDA'):
    try:
        prefs.compute_device_type = dev_type
        for d in prefs.devices:
            d.use = True
        print(f'Cycles device enabled: {dev_type}')
        break
    except Exception:
        continue
"""
open('/content/enable_gpu.py', 'w').write(enable_gpu_script)
subprocess.run(['blender', '-b', '--python', '/content/enable_gpu.py'], check=True)

### 3. AGENT WEB 接続トンネルとワーカーの起動
次のセルを実行すると、一時HTTPS URLと接続キーが表示されます。AGENT WEBのColab画面に登録してください。

In [ ]:
"""Run in Colab after installing Blender. Only authenticated render jobs are exposed."""
import base64
import hashlib
import hmac
import json
import re
import secrets
import subprocess
import threading
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path


def start_worker(port=8765):
    token = secrets.token_urlsafe(32)
    jobs = {}
    assets = {}
    clips = {}
    lock = threading.Lock()
    instance_id = str(uuid.uuid4())
    workdir = Path('/content/aw_blender')
    workdir.mkdir(parents=True, exist_ok=True)
    try:
        gpu_name = subprocess.run(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            capture_output=True, text=True, timeout=10).stdout.strip() or 'CPU'
    except Exception:
        gpu_name = 'CPU'

    def build_render_script(params):
        engine_id = 'CYCLES' if params['engine'] == 'CYCLES' else 'BLENDER_EEVEE_NEXT'
        lines = [
            'import bpy',
            'scene = bpy.context.scene',
            'try:',
            "    prefs = bpy.context.preferences.addons['cycles'].preferences",
            '    prefs.get_devices()',
            "    for dev_type in ('OPTIX', 'CUDA'):",
            '        try:',
            '            prefs.compute_device_type = dev_type',
            '            for d in prefs.devices:',
            '                d.use = True',
            "            scene.cycles.device = 'GPU'",
            '            break',
            '        except Exception:',
            '            continue',
            'except Exception:',
            '    pass',
            f"scene.render.engine = '{engine_id}'",
            f"scene.render.image_settings.file_format = '{params['format']}'",
            'scene.render.resolution_percentage = 100',
        ]
        if params['engine'] == 'CYCLES' and params.get('samples'):
            lines.append(f"scene.cycles.samples = {params['samples']}")
        if params.get('resolution_x'):
            lines.append(f"scene.render.resolution_x = {params['resolution_x']}")
            lines.append(f"scene.render.resolution_y = {params['resolution_y']}")
        return '\n'.join(lines) + '\n'

    def run_render(job_id, params):
        try:
            job_dir = workdir / job_id
            job_dir.mkdir(parents=True, exist_ok=True)
            blend_path = job_dir / 'input.blend'
            blend_path.write_bytes(assets[params['asset']])
            (job_dir / 'render.py').write_text(build_render_script(params))
            total = params['frame_end'] - params['frame_start'] + 1
            cmd = ['blender', '-b', str(blend_path), '--python', str(job_dir / 'render.py'),
                   '-o', str(job_dir / 'frame_')]
            if total > 1:
                cmd += ['-s', str(params['frame_start']), '-e', str(params['frame_end']), '-a']
            else:
                cmd += ['-f', str(params['frame_start'])]
            proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
            while proc.poll() is None:
                done = len([f for f in job_dir.glob('frame_*.*')
                            if f.suffix.lower() in ('.png', '.jpg', '.jpeg')])
                with lock:
                    if job_id in jobs:
                        jobs[job_id]['progress'] = min(99, round(done / total * 100))
                threading.Event().wait(2)
            files = sorted(f for f in job_dir.glob('frame_*.*')
                           if f.suffix.lower() in ('.png', '.jpg', '.jpeg'))
            if proc.returncode != 0 or len(files) != total:
                raise RuntimeError(
                    f'Blender exited with code {proc.returncode}; '
                    f'{len(files)}/{total} frames rendered')
            encoded = [f.read_bytes() for f in files]
            with lock:
                clips[job_id] = encoded
                update = {'status': 'completed', 'progress': 100,
                          'frame_count': len(encoded)}
                if total == 1:
                    update['image'] = base64.b64encode(encoded[0]).decode()
                jobs[job_id].update(**update)
        except Exception as exc:
            with lock:
                jobs[job_id].update(status='failed', error=str(exc)[:2000])

    class Handler(BaseHTTPRequestHandler):
        def log_message(self, *args):
            pass

        def respond(self, status, body):
            raw = json.dumps(body).encode()
            self.send_response(status)
            self.send_header('Content-Type', 'application/json')
            self.send_header('Content-Length', str(len(raw)))
            self.end_headers()
            self.wfile.write(raw)

        def authorized(self):
            if not hmac.compare_digest(self.headers.get('Authorization', ''), 'Bearer ' + token):
                self.respond(401, {'error': 'Unauthorized'})
                return False
            return True

        def do_GET(self):
            if not self.authorized(): return
            if self.path == '/status':
                with lock:
                    busy = any(j['status'] == 'running' for j in jobs.values())
                self.respond(200, {'gpu': gpu_name,
                    'model': 'Blender 4.2 LTS', 'busy': busy, 'protocol': 1,
                    'instance_id': instance_id,
                    'capabilities': {'reference_image': False, 'text_to_image': False,
                                     'motion': False, 'render': True}})
            elif re.fullmatch(r'/jobs/[0-9a-f-]+/frames/[0-9]+', self.path):
                parts = self.path.split('/'); job_id = parts[2]; index = int(parts[4])
                with lock:
                    clip = clips.get(job_id, [])
                    raw = clip[index] if 0 <= index < len(clip) else None
                    job = dict(jobs.get(job_id, {}))
                if raw is None: return self.respond(404, {'error': 'Frame not available'})
                ctype = 'image/jpeg' if job.get('format') == 'JPEG' else 'image/png'
                self.send_response(200); self.send_header('Content-Type', ctype)
                self.send_header('Content-Length', str(len(raw))); self.end_headers()
                self.wfile.write(raw)
            elif self.path.startswith('/jobs/'):
                with lock:
                    job = dict(jobs.get(self.path.split('/')[-1], {}))
                self.respond(200 if job else 404, job or {'error': 'Unknown job'})
            else:
                self.respond(404, {'error': 'Not found'})

        def do_POST(self):
            if not self.authorized(): return
            if self.path.startswith('/assets/'):
                try:
                    digest = self.path.split('/')[-1]
                    if not re.fullmatch('[a-f0-9]{64}', digest): raise ValueError('Invalid asset ID')
                    length = int(self.headers.get('Content-Length', 0))
                    if not 0 < length <= 256 * 1024 * 1024: raise ValueError('Invalid asset size')
                    raw = self.rfile.read(length)
                    if hashlib.sha256(raw).hexdigest() != digest: raise ValueError('Asset checksum mismatch')
                    with lock:
                        if digest not in assets and len(assets) >= 8:
                            protected = {j.get('asset') for j in jobs.values()
                                         if j['status'] == 'running'}
                            for old in list(assets):
                                if old not in protected: del assets[old]; break
                        assets[digest] = raw
                    return self.respond(200, {'sha256': digest})
                except ValueError: return self.respond(400, {'error': 'Invalid blend asset'})
            if self.path != '/jobs': return self.respond(404, {'error': 'Not found'})
            try:
                length = int(self.headers.get('Content-Length', 0))
                if not 0 < length <= 32768: raise ValueError('Invalid request size')
                params = json.loads(self.rfile.read(length))
                job_id = str(uuid.UUID(params['id']))
                asset = params.get('asset')
                if not isinstance(asset, str) or asset not in assets:
                    raise ValueError('Blend file unavailable')
                engine = params.get('engine', 'CYCLES')
                if engine not in ('CYCLES', 'EEVEE'): raise ValueError('Invalid engine')
                frame_start = params.get('frame_start', 1)
                frame_end = params.get('frame_end', frame_start)
                for v in (frame_start, frame_end):
                    if type(v) is not int or not 1 <= v <= 100000:
                        raise ValueError('Invalid frame number')
                if frame_end < frame_start or frame_end - frame_start + 1 > 250:
                    raise ValueError('Invalid frame range')
                fmt = params.get('format', 'PNG')
                if fmt not in ('PNG', 'JPEG'): raise ValueError('Invalid format')
                samples = params.get('samples')
                if samples is not None and (type(samples) is not int or not 1 <= samples <= 4096):
                    raise ValueError('Invalid samples')
                res_x, res_y = params.get('resolution_x'), params.get('resolution_y')
                if (res_x is None) != (res_y is None):
                    raise ValueError('Invalid resolution')
                if res_x is not None:
                    for v in (res_x, res_y):
                        if type(v) is not int or not 64 <= v <= 4096:
                            raise ValueError('Invalid resolution')
                job_params = {'asset': asset, 'engine': engine,
                              'frame_start': frame_start, 'frame_end': frame_end,
                              'format': fmt, 'samples': samples,
                              'resolution_x': res_x, 'resolution_y': res_y}
                with lock:
                    if job_id in jobs: return self.respond(200, {'id': job_id})
                    if any(j['status'] == 'running' for j in jobs.values()):
                        return self.respond(409, {'error': 'GPU is busy'})
                    jobs[job_id] = {'id': job_id, 'status': 'running', 'progress': 0,
                                    'mode': 'render', 'format': fmt, 'asset': asset}
                threading.Thread(target=run_render, args=(job_id, job_params), daemon=True).start()
                self.respond(202, {'id': job_id})
            except (ValueError, KeyError, TypeError) as exc:
                self.respond(400, {'error': str(exc)})

    server = ThreadingHTTPServer(('127.0.0.1', port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, token

import os
import re
import subprocess
import time
import urllib.request
from pathlib import Path

if "aw_tunnel" in globals():
    aw_tunnel.terminate()
    aw_tunnel.wait(timeout=10)
if "aw_server" in globals():
    aw_server.shutdown()
    aw_server.server_close()
aw_server, aw_token = start_worker()
binary = Path("/content/cloudflared")
if not binary.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        binary,
    )
    binary.chmod(0o700)
log = open("/content/agent-web-tunnel.log", "w+")
aw_tunnel = subprocess.Popen(
    [str(binary), "tunnel", "--url", "http://127.0.0.1:8765", "--no-autoupdate"],
    stdout=log, stderr=subprocess.STDOUT,
)
aw_url = None
for _ in range(60):
    time.sleep(1)
    log.seek(0)
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log.read())
    if match:
        aw_url = match.group(0)
        break
    if aw_tunnel.poll() is not None:
        break
if not aw_url:
    raise RuntimeError("トンネルを起動できませんでした。/content/agent-web-tunnel.log を確認してください。")
print("agent-webのColab画面に登録してください")
print("接続URL:", aw_url)
print("接続キー:", aw_token)


作業終了時は次のセルで接続を止め、**ランタイム → 接続解除してランタイムを削除**でGPUを解放してください。

In [ ]:
aw_tunnel.terminate()